In [1]:
## Parallel programming imports
import ipyparallel as ipp
from mpi4py import MPI
cluster = ipp.Cluster(engines = "mpi", n = 10)
rc = cluster.start_and_connect_sync()


Starting 10 engines with <class 'ipyparallel.cluster.launcher.MPIEngineSetLauncher'>


  0%|          | 0/10 [00:00<?, ?engine/s]

In [19]:
%%px 

import gmsh
import os
import time
import numpy as np
from dolfinx.io import gmshio
from mpi4py import MPI
import pyvista as pv
from dolfinx.plot import vtk_mesh
import cv2
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import pyvista
from dolfinx.plot import vtk_mesh
from dolfinx.mesh import create_submesh
from dolfinx.fem import form, assemble_scalar, Constant
from ufl import dx

gmsh.initialize(sys.argv)


[stderr:4] Warning : Gmsh has aleady been initialized


[stderr:0] Warning : Gmsh has aleady been initialized


[stderr:1] Warning : Gmsh has aleady been initialized


[stderr:2] Warning : Gmsh has aleady been initialized


[stderr:5] Warning : Gmsh has aleady been initialized


[stderr:6] Warning : Gmsh has aleady been initialized


[stderr:3] Warning : Gmsh has aleady been initialized


[stderr:7] Warning : Gmsh has aleady been initialized


[stderr:9] Warning : Gmsh has aleady been initialized


[stderr:8] Warning : Gmsh has aleady been initialized


In [ ]:
%%px

def load_first_layer(msh_file):
    # Convert mesh if needed and import
    print(type(msh_file))
    mesh_, cell_tags, facet_tags = gmshio.read_from_msh(msh_file, comm = MPI.COMM_WORLD, rank=0, gdim=3)
    return mesh_, cell_tags, facet_tags

In [ ]:
%%px

def write_partitioned_mesh(filename: Path):
    import subprocess
    from mpi4py import MPI
    import dolfinx
    import adios4dolfinx

    mesh, cell_tags, facet_tags = load_first_layer(os.path.join("layers", "layer_42.msh"))
    
    # Write mesh checkpoint
    adios4dolfinx.write_mesh(filename, mesh, engine="BP4", store_partition_info=True)
    adios4dolfinx.write_meshtags(filename, mesh, cell_tags, engine="BP4", meshtag_name = "cells")
    adios4dolfinx.write_meshtags(filename, mesh, facet_tags, engine="BP4", meshtag_name = "facets")
    # Inspect checkpoint on rank 0 with `bpls`
    if mesh.comm.rank == 0:
        output = subprocess.run(["bpls", "-a", "-l", filename], capture_output=True)
        print(output.stdout.decode("utf-8"))

In [3]:
%%px 

def read_partitioned_mesh(filename: Path, read_from_partition: bool = True):
    from mpi4py import MPI

    import adios4dolfinx

    prefix = f"{MPI.COMM_WORLD.rank + 1}/{MPI.COMM_WORLD.size}: "
    try:
        mesh = adios4dolfinx.read_mesh(
            filename, comm=MPI.COMM_WORLD, engine="BP4", read_from_partition=read_from_partition
        )
        cell_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "cells", engine="BP4")
        facet_tags = adios4dolfinx.read_meshtags(filename, mesh, meshtag_name = "facets", engine="BP4")

        tdim = mesh.topology.dim
        mesh.topology.create_connectivity(tdim - 1, tdim)

        print(f"{prefix} Mesh: {mesh.name} read successfully with {read_from_partition=}")
    except ValueError as e:
        print(f"{prefix} Caught exception: ", e)

    return mesh, cell_tags, facet_tags



In [4]:
%%px 

def plot_partitioned_mesh(mesh_):
    comm = MPI.COMM_WORLD
    rank = comm.rank

    if rank == 0:
        plotter = pyvista.Plotter(off_screen=True)
        #plotter.add_title('Plot Title', font='courier', color='k', font_size=40) #not sure yet whether to add a title or not 

    cells, types, x = vtk_mesh(mesh_)
    cells_all = comm.gather(cells, root=0)
    types_all = comm.gather(types, root=0)
    x_all = comm.gather(x, root=0)
    print("Gathering data completed")

    if (rank == 0):
        for cells_i, types_i, x_i in zip(cells_all, types_all, x_all):
            grid = pyvista.UnstructuredGrid(cells_i, types_i, x_i)
            plotter.add_mesh(grid, show_edges=True)
            print("Added mesh to plotter")
        plotter.show()
        #plotter.close()  # Finish writing the GIF



In [ ]:
%%px 

import time 

mesh_file = Path("./partitioned_meshes/00005.bp")
top_marker = 13
#write_partitioned_mesh(mesh_file)

start_time = time.time()
mesh1, cell_tags1, facet_tags1 = read_partitioned_mesh(mesh_file, True)
end_time = time.time()
#print(f"Time taken to read mesh: {(end_time - start_time)*1000} ms")
#print(f"Loaded mesh with {mesh1.topology.index_map(3).size_local} cells.")
#print(len(mesh.cell_tags.values), "DOFs for this process")

submesh, entity_map, vertex_map, node_map = create_submesh(mesh1, dim = 2, entities = facet_tags1.find(top_marker))
area = assemble_scalar(form(Constant(submesh, 1.0)*dx))

total_area = MPI.COMM_WORLD.gather(area, root=0)
total_area = np.sum(np.array(total_area))


if MPI.COMM_WORLD.rank == 0: 
    print(f"Area of submesh: {area}")
    print(f"Total area of submesh: {total_area}")
    print(f"{np.sum(total_area*1000*1000)} mm^2" )

%px:   0%|          | 0/10 [00:00<?, ?tasks/s]

[stdout:1] 2/10:  Mesh: mesh read successfully with read_from_partition=True


[stdout:5] 6/10:  Mesh: mesh read successfully with read_from_partition=True


[stdout:7] 8/10:  Mesh: mesh read successfully with read_from_partition=True


[stdout:3] 4/10:  Mesh: mesh read successfully with read_from_partition=True


...

[stdout:9] 10/10:  Mesh: mesh read successfully with read_from_partition=True


[stdout:8] 9/10:  Mesh: mesh read successfully with read_from_partition=True


[stdout:4] 5/10:  Mesh: mesh read successfully with read_from_partition=True


[stdout:0] 1/10:  Mesh: mesh read successfully with read_from_partition=True
Area of submesh: 8.304277931591074e-05
Total area of submesh: 0.0008316314552660959
831.6314552660958 mm^2


[stdout:6] 7/10:  Mesh: mesh read successfully with read_from_partition=True


In [ ]:
%%px 

plot_partitioned_mesh(mesh1)